
# Projet B1 : Équilibrage de chaîne d'assemblage (SALBP)

Groupe : Ilias Kalalou - Kaelan Grall

EPITA SCIA - Programmation par Contraintes 2026

## Sommaire

1. Introduction et formalisation
2. Instance d'exemple
3. SALBP-1 (minimiser le nombre de stations) avec CP-SAT
4. SALBP-2 (minimiser le cycle time) avec CP-SAT
5. Heuristique Ranked Positional Weight (RPW)
6. Modèle PLNE (PuLP)
7. Benchmark comparatif
8. Extensions (pistes d'évolution)
9. Conclusion

## 1. Introduction et formalisation

Le **Simple Assembly Line Balancing Problem (SALBP)** consiste à répartir un ensemble de tâches sur les stations d'une chaîne d'assemblage.

### Données du problème

| Élément | Notation | Description |
|---------|----------|-------------|
| Tâches | T = {1, ..., n} | Tâches élémentaires |
| Durée | t_i | Temps d'exécution de la tâche i |
| Précédences | P | Ensemble de couples (i, j) : i doit finir avant j |
| Cycle time | C | Temps maximum alloué à chaque station |

### Variantes

| Variante | Fixé | Minimisé |
|----------|------|----------|
| SALBP-1 | Cycle time C | Nombre de stations m |
| SALBP-2 | Nombre de stations m | Cycle time C |

### Contraintes

1. Chaque tâche est assignée à exactement une station
2. Si (i, j) est une précédence, alors station(i) <= station(j)
3. La somme des durées des tâches d'une station ne dépasse pas C

In [ ]:
%pip install -q ortools pulp networkx matplotlib

import importlib
importlib.invalidate_caches()

import time
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from ortools.sat.python import cp_model
import pulp

print("Imports OK")

## 2. Instance d'exemple

Nous utilisons une instance jouet de 11 tâches avec un graphe de précédence acyclique. Cette instance est inspirée de l'exemple classique de Bowman.

In [ ]:
# Instance jouet : 11 tâches
tasks = list(range(11))
durations = {0: 6, 1: 5, 2: 4, 3: 5, 4: 3, 5: 6, 6: 4, 7: 7, 8: 5, 9: 6, 10: 7}

# Précédences : liste d'arêtes (i -> j) signifie i avant j
precedences = [
    (0, 1), (0, 2), (1, 3), (2, 3), (2, 4),
    (3, 5), (4, 5), (4, 6), (5, 7), (6, 7),
    (7, 8), (7, 9), (8, 10), (9, 10)
]

cycle_time = 12  # temps maximum par station pour SALBP-1

print(f"Nombre de tâches : {len(tasks)}")
print(f"Durée totale : {sum(durations.values())}")
print(f"Cycle time : {cycle_time}")
print(f"Borne inférieure du nombre de stations : {-(-sum(durations.values()) // cycle_time)}")

In [ ]:
# Visualisation du graphe de précédence
G = nx.DiGraph()
for t in tasks:
    G.add_node(t, duration=durations[t])
G.add_edges_from(precedences)

pos = nx.spring_layout(G, seed=42, k=1.5)

fig, ax = plt.subplots(figsize=(10, 6))
node_labels = {t: f"{t}\n({durations[t]})" for t in tasks}
nx.draw(G, pos, ax=ax, with_labels=True, labels=node_labels,
        node_color='#4ECDC4', node_size=1000, font_size=9,
        font_weight='bold', edge_color='gray', arrows=True,
        arrowsize=15, edgecolors='black')
ax.set_title("Graphe de précédence (tâche / durée)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. SALBP-1 avec CP-SAT

### Modélisation

Variables :
- `station[i]` : numéro de station de la tâche i, dans [0, m_max - 1]
- `n_stations` : nombre de stations utilisées (à minimiser)

Contraintes :
- Précédence : `station[i] <= station[j]` pour (i, j) dans P
- `station[i] < n_stations` pour toute tâche i
- Temps cumulé par station <= cycle_time (via réifications)

Objectif : minimiser `n_stations`.

In [ ]:
def solve_salbp1_cpsat(tasks, durations, precedences, cycle_time, time_limit=30):
    """Minimise le nombre de stations pour un cycle time donné."""
    m_max = len(tasks)  # borne triviale : une station par tâche
    model = cp_model.CpModel()

    # Variables
    station = {i: model.new_int_var(0, m_max - 1, f'station_{i}') for i in tasks}
    n_stations = model.new_int_var(1, m_max, 'n_stations')

    # Précédences
    for (i, j) in precedences:
        model.add(station[i] <= station[j])

    # station[i] < n_stations
    for i in tasks:
        model.add(station[i] < n_stations)

    # Capacité cycle time par station (via booléens d'assignation)
    assign = {(i, s): model.new_bool_var(f'a_{i}_{s}') for i in tasks for s in range(m_max)}
    for i in tasks:
        model.add_exactly_one(assign[i, s] for s in range(m_max))
        for s in range(m_max):
            model.add(station[i] == s).only_enforce_if(assign[i, s])
            model.add(station[i] != s).only_enforce_if(assign[i, s].Not())

    for s in range(m_max):
        model.add(sum(durations[i] * assign[i, s] for i in tasks) <= cycle_time)

    # Objectif
    model.minimize(n_stations)

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = time_limit
    status = solver.solve(model)

    if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        assignment = {i: solver.value(station[i]) for i in tasks}
        return {
            'n_stations': solver.value(n_stations),
            'assignment': assignment,
            'optimal': status == cp_model.OPTIMAL,
            'time_ms': solver.wall_time * 1000,
        }
    return None


result_s1 = solve_salbp1_cpsat(tasks, durations, precedences, cycle_time)
print(f"SALBP-1 : {result_s1['n_stations']} stations (optimal={result_s1['optimal']}, {result_s1['time_ms']:.0f} ms)")
for i in tasks:
    print(f"  tâche {i} -> station {result_s1['assignment'][i]}")

In [ ]:
# Visualisation Gantt des stations
def draw_stations(assignment, durations, title):
    stations = {}
    for i, s in assignment.items():
        stations.setdefault(s, []).append(i)

    fig, ax = plt.subplots(figsize=(12, max(3, len(stations))))
    colors = plt.cm.Set3(np.linspace(0, 1, len(stations)))

    for s_idx, (s, task_list) in enumerate(sorted(stations.items())):
        x = 0
        for t in task_list:
            ax.barh(s, durations[t], left=x, color=colors[s_idx],
                    edgecolor='black', linewidth=1)
            ax.text(x + durations[t] / 2, s, f"T{t}\n({durations[t]})",
                    ha='center', va='center', fontsize=9, fontweight='bold')
            x += durations[t]

    ax.set_yticks(sorted(stations.keys()))
    ax.set_yticklabels([f"Station {s}" for s in sorted(stations.keys())])
    ax.set_xlabel('Temps', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axvline(x=cycle_time, color='red', linestyle='--', label=f'Cycle time = {cycle_time}')
    ax.legend(loc='lower right')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

draw_stations(result_s1['assignment'], durations,
              f"SALBP-1 optimal : {result_s1['n_stations']} stations (CP-SAT)")

## 4. SALBP-2 avec CP-SAT

Variante duale : on fixe le nombre de stations `m` et on minimise le cycle time.

In [ ]:
def solve_salbp2_cpsat(tasks, durations, precedences, n_stations, time_limit=30):
    """Minimise le cycle time pour un nombre de stations fixé."""
    model = cp_model.CpModel()

    station = {i: model.new_int_var(0, n_stations - 1, f's_{i}') for i in tasks}
    cycle_var = model.new_int_var(max(durations.values()), sum(durations.values()), 'cycle')

    for (i, j) in precedences:
        model.add(station[i] <= station[j])

    assign = {(i, s): model.new_bool_var(f'a_{i}_{s}') for i in tasks for s in range(n_stations)}
    for i in tasks:
        model.add_exactly_one(assign[i, s] for s in range(n_stations))
        for s in range(n_stations):
            model.add(station[i] == s).only_enforce_if(assign[i, s])
            model.add(station[i] != s).only_enforce_if(assign[i, s].Not())

    for s in range(n_stations):
        model.add(sum(durations[i] * assign[i, s] for i in tasks) <= cycle_var)

    model.minimize(cycle_var)

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = time_limit
    status = solver.solve(model)

    if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        return {
            'cycle_time': solver.value(cycle_var),
            'assignment': {i: solver.value(station[i]) for i in tasks},
            'optimal': status == cp_model.OPTIMAL,
            'time_ms': solver.wall_time * 1000,
        }
    return None


# On fixe m = n_stations trouvé par SALBP-1 et on minimise le cycle
m_fixed = result_s1['n_stations']
result_s2 = solve_salbp2_cpsat(tasks, durations, precedences, m_fixed)
print(f"SALBP-2 ({m_fixed} stations) : cycle time = {result_s2['cycle_time']} (optimal={result_s2['optimal']}, {result_s2['time_ms']:.0f} ms)")

## 5. Heuristique Ranked Positional Weight (RPW)

RPW est une heuristique gloutonne classique pour SALBP-1.

Pour chaque tâche i, on calcule son poids positionnel :

**RPW(i) = t_i + somme des t_j pour tout j successeur (transitif) de i**

On trie les tâches par RPW décroissant, puis on les assigne dans cet ordre à la première station ayant assez de temps libre et dont les prédécesseurs sont déjà assignés à une station <= s.

In [ ]:
def compute_rpw(tasks, durations, precedences):
    G = nx.DiGraph()
    G.add_nodes_from(tasks)
    G.add_edges_from(precedences)
    rpw = {}
    for i in tasks:
        successors = nx.descendants(G, i)
        rpw[i] = durations[i] + sum(durations[j] for j in successors)
    return rpw


def solve_salbp1_rpw(tasks, durations, precedences, cycle_time):
    """Heuristique RPW pour SALBP-1."""
    rpw = compute_rpw(tasks, durations, precedences)
    order = sorted(tasks, key=lambda i: -rpw[i])

    preds = {i: set() for i in tasks}
    for (a, b) in precedences:
        preds[b].add(a)

    assignment = {}
    stations_load = []  # charge de chaque station

    for i in order:
        min_station = max((assignment[p] for p in preds[i] if p in assignment), default=-1) + 1
        placed = False
        for s in range(min_station, len(stations_load)):
            if stations_load[s] + durations[i] <= cycle_time:
                assignment[i] = s
                stations_load[s] += durations[i]
                placed = True
                break
        if not placed:
            assignment[i] = len(stations_load)
            stations_load.append(durations[i])

    return {'n_stations': len(stations_load), 'assignment': assignment}


start = time.time()
result_rpw = solve_salbp1_rpw(tasks, durations, precedences, cycle_time)
t_rpw = (time.time() - start) * 1000
print(f"RPW : {result_rpw['n_stations']} stations ({t_rpw:.2f} ms)")

## 6. Modèle PLNE (PuLP)

Variables binaires `x[i][s] = 1` si la tâche i est à la station s. Le modèle est quasi identique au CP-SAT mais formulé en programmation linéaire en nombres entiers, résolu par un solveur classique (CBC par défaut).

In [ ]:
def solve_salbp1_plne(tasks, durations, precedences, cycle_time, time_limit=30):
    m_max = len(tasks)
    prob = pulp.LpProblem("SALBP1", pulp.LpMinimize)

    x = {(i, s): pulp.LpVariable(f"x_{i}_{s}", cat="Binary")
         for i in tasks for s in range(m_max)}
    y = {s: pulp.LpVariable(f"y_{s}", cat="Binary") for s in range(m_max)}

    prob += pulp.lpSum(y[s] for s in range(m_max))

    for i in tasks:
        prob += pulp.lpSum(x[i, s] for s in range(m_max)) == 1

    for s in range(m_max):
        prob += pulp.lpSum(durations[i] * x[i, s] for i in tasks) <= cycle_time * y[s]

    for (i, j) in precedences:
        prob += pulp.lpSum(s * x[i, s] for s in range(m_max)) <= \
                pulp.lpSum(s * x[j, s] for s in range(m_max))

    solver = pulp.PULP_CBC_CMD(msg=0, timeLimit=time_limit)
    prob.solve(solver)

    if pulp.LpStatus[prob.status] not in ('Optimal', 'Not Solved'):
        return None

    assignment = {}
    for i in tasks:
        for s in range(m_max):
            if pulp.value(x[i, s]) > 0.5:
                assignment[i] = s
                break

    return {
        'n_stations': int(pulp.value(prob.objective)),
        'assignment': assignment,
        'time_ms': prob.solutionTime * 1000,
    }


result_plne = solve_salbp1_plne(tasks, durations, precedences, cycle_time)
print(f"PLNE : {result_plne['n_stations']} stations ({result_plne['time_ms']:.0f} ms)")

## 7. Benchmark comparatif

Comparaison des trois approches sur notre instance jouet.

In [ ]:
print(f"{'Approche':<15} {'Stations':>10} {'Temps (ms)':>12} {'Optimal':>10}")
print("-" * 50)
print(f"{'CP-SAT':<15} {result_s1['n_stations']:>10} {result_s1['time_ms']:>12.1f} {'Oui' if result_s1['optimal'] else 'Non':>10}")
print(f"{'PLNE (CBC)':<15} {result_plne['n_stations']:>10} {result_plne['time_ms']:>12.1f} {'Oui':>10}")
print(f"{'RPW (heur.)':<15} {result_rpw['n_stations']:>10} {t_rpw:>12.2f} {'?':>10}")

## 8. Pistes d'extension

### 8.1 Benchmarks Otto et al. (2013)

Les instances de référence sont disponibles sur assembly-line-balancing.de. Le format `.alb` contient :
- nombre de tâches
- cycle time
- durées des tâches
- précédences

Il faudra implémenter un parseur et comparer CP-SAT, PLNE et RPW sur j30, j60, j120, etc.

### 8.2 Variante multi-modèles (MMALBP)

Étendre le modèle pour mélanger plusieurs produits sur la même ligne, avec des durées et précédences différentes par produit.

### 8.3 Multi-objectifs

Minimiser simultanément le nombre de stations et le déséquilibre de charge (écart-type des charges par station) via front de Pareto.

### 8.4 Visualisation interactive

Intégrer une petite UI Streamlit pour charger une instance, paramétrer le cycle time et visualiser le résultat.

## 9. Conclusion

Ce notebook fournit une base fonctionnelle pour le SALBP avec trois approches :
- **CP-SAT** (exact, optimal)
- **PLNE via PuLP/CBC** (exact, autre formulation)
- **RPW** (heuristique gloutonne)

Prochaines étapes :
1. Parser et charger les instances Otto
2. Benchmarker à grande échelle (n=30, 60, 120)
3. Implémenter SALBP-2 sur les mêmes instances
4. Ajouter la variante multi-modèles
5. Préparer l'UI et les slides de soutenance